In [ ]:
import ipaddress
import json
import math
import re
import socket
import time
import urllib.error
import urllib.parse
import urllib.request
import pandas as pd
import websocket


class CollectionError(RuntimeError):
    pass


class _CollectionNoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, request, response, code, message, headers, new_url):
        raise CollectionError("A redirect was refused; use the endpoint's direct address.") from None


def _collection_time(value):
    try:
        result = pd.Timestamp(value)
        if pd.isna(result) or result.tzinfo is None:
            raise ValueError
        return result.tz_convert("UTC")
    except Exception:
        raise CollectionError("A valid date and time with a timezone is required.") from None


def _collection_window(start, end, chunk_days):
    start, end = _collection_time(start), _collection_time(end)
    if not start < end or end - start > pd.Timedelta(days=3660):
        raise CollectionError("Choose an increasing collection period of at most ten years.")
    if type(chunk_days) is not int or not 1 <= chunk_days <= 31:
        raise CollectionError("Choose a chunk length between one and 31 days.")
    return start, end


def _collection_endpoint(endpoint, token):
    try:
        address = urllib.parse.urlsplit(endpoint)
        if address.scheme not in {"http", "https", "ws", "wss"}:
            raise ValueError
        if not address.hostname or address.username or address.password or address.query or address.fragment:
            raise ValueError
        if address.path not in {"", "/", "/api", "/api/", "/api/websocket"}:
            raise ValueError
        address.port
        if not isinstance(token, str) or not token.strip() or any(c.isspace() for c in token):
            raise ValueError
        if address.scheme in {"http", "ws"}:
            host = address.hostname.lower()
            try:
                number = ipaddress.ip_address(host)
                allowed = number.is_private or number.is_loopback or number in ipaddress.ip_network("100.64.0.0/10")
            except ValueError:
                allowed = "." not in host or host.endswith(".ts.net") or host.endswith(".local")
            if not allowed:
                raise CollectionError("Use HTTPS outside a private network or tailnet.")
        scheme = "https" if address.scheme in {"https", "wss"} else "http"
        return urllib.parse.urlunsplit((scheme, address.netloc, "", "", ""))
    except CollectionError:
        raise
    except Exception:
        raise CollectionError("Enter a direct Home Assistant endpoint and a non-empty access token.") from None


def _collection_channels(channels, key):
    if not isinstance(channels, dict) or not 1 <= len(channels) <= 100:
        raise CollectionError("Choose between one and 100 explicitly named channels.")
    for alias, details in channels.items():
        if not isinstance(alias, str) or not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,63}", alias):
            raise CollectionError("Use short anonymous channel labels containing letters, digits and underscores.")
        if not isinstance(details, dict) or not isinstance(details.get(key), str):
            raise CollectionError("Each channel needs an explicit source identifier.")
        if not re.fullmatch(r"[a-z0-9_]+[.:][a-z0-9_]+", details[key]):
            raise CollectionError("A channel source identifier is not in the expected format.")
        attribute = details.get("attribute")
        if attribute is not None and (not isinstance(attribute, str) or not re.fullmatch(r"[a-z][a-z0-9_]{0,63}", attribute)):
            raise CollectionError("Choose one named numeric attribute, or collect the state.")


def _collection_json(address, headers=None, timeout=30, attempts=3):
    if type(attempts) is not int or not 1 <= attempts <= 3 or not 1 <= timeout <= 60:
        raise CollectionError("Use one to three attempts and a timeout of one to 60 seconds.")
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), _CollectionNoRedirect())
    for attempt in range(attempts):
        try:
            request = urllib.request.Request(address, headers=headers or {}, method="GET")
            with opener.open(request, timeout=timeout) as response:
                if response.status != 200:
                    raise CollectionError("The collection service did not return a successful response.")
                payload = response.read(33554433)
            if len(payload) > 33554432:
                raise CollectionError("A response exceeded 32 MiB; shorten the collection chunks.")
            return json.loads(payload)
        except CollectionError:
            raise
        except urllib.error.HTTPError as error:
            if error.code in {401, 403}:
                raise CollectionError("Access was refused; check the runtime token and its read permissions.") from None
            if error.code not in {408, 429, 500, 502, 503, 504} or attempt == attempts - 1:
                raise CollectionError("The collection request failed; check service availability and input choices.") from None
        except (urllib.error.URLError, TimeoutError, socket.timeout, OSError):
            if attempt == attempts - 1:
                raise CollectionError("The collection endpoint could not be reached within the retry limit.") from None
        except Exception:
            raise CollectionError("The collection service returned an unreadable response.") from None
        time.sleep(min(2 ** attempt, 4))


def _collection_state(raw_state):
    if raw_state is None:
        return "missing", float("nan")
    if isinstance(raw_state, bool) or not isinstance(raw_state, (str, int, float)):
        raise CollectionError("A history observation has an unsupported state type.")
    try:
        value = float(raw_state)
        if math.isfinite(value):
            return str(raw_state), value
    except (ValueError, TypeError):
        pass
    known_states = {"unknown", "unavailable", "on", "off", "heat", "cool", "auto", "heating", "cooling", "idle", "dry", "fan_only", "heat_cool", "open", "closed", "standby"}
    return (raw_state if raw_state in known_states else "other"), float("nan")


def collect_home_assistant_history(endpoint, token, channels, start, end, chunk_days=1):
    base = _collection_endpoint(endpoint, token)
    _collection_channels(channels, "entity_id")
    start, end = _collection_window(start, end, chunk_days)
    records = []
    for alias, details in channels.items():
        cursor = start
        while cursor < end:
            stop = min(cursor + pd.Timedelta(days=chunk_days), end)
            parameters = {"filter_entity_id": details["entity_id"], "end_time": stop.isoformat(), "significant_changes_only": "0", "skip_initial_state": "1"}
            if details.get("attribute") is None:
                parameters["no_attributes"] = "1"
            address = base + "/api/history/period/" + urllib.parse.quote(cursor.isoformat(), safe="") + "?" + urllib.parse.urlencode(parameters)
            payload = _collection_json(address, {"Authorization": "Bearer " + token, "Accept": "application/json"})
            if not isinstance(payload, list) or any(not isinstance(group, list) for group in payload):
                raise CollectionError("History returned an unexpected record structure.")
            for group in payload:
                for observation in group:
                    if not isinstance(observation, dict) or observation.get("entity_id") != details["entity_id"] or "state" not in observation:
                        raise CollectionError("History returned an unexpected channel or observation.")
                    stamp = _collection_time(observation.get("last_updated", observation.get("last_changed")))
                    if not start <= stamp < end:
                        continue
                    if details.get("attribute") is None:
                        state, value = _collection_state(observation["state"])
                    else:
                        attributes = observation.get("attributes", {})
                        if not isinstance(attributes, dict):
                            raise CollectionError("History returned invalid attribute data.")
                        raw_value = attributes.get(details["attribute"])
                        if observation["state"] in {"unknown", "unavailable"}:
                            raw_value = observation["state"]
                        state, value = _collection_state(raw_value)
                        if state not in {"missing", "unknown", "unavailable"} and not math.isfinite(value):
                            raise CollectionError("The selected attribute is not numeric.")
                    records.append({"channel": alias, "timestamp": stamp, "state": state, "value": value})
            cursor = stop
    result = pd.DataFrame(records, columns=["channel", "timestamp", "state", "value"])
    result["timestamp"] = pd.to_datetime(result["timestamp"], utc=True)
    result["value"] = pd.to_numeric(result["value"], errors="raise")
    result = result.drop_duplicates().sort_values(["channel", "timestamp"], kind="stable").reset_index(drop=True)
    result["same_time_conflict"] = result.duplicated(["channel", "timestamp"], keep=False)
    return result


In [ ]:
def _collection_ws_message(connection):
    raw = connection.recv()
    if not isinstance(raw, (str, bytes)) or len(raw) > 33554432:
        raise CollectionError("A statistics response exceeded the allowed size or type.")
    message = json.loads(raw)
    if not isinstance(message, dict):
        raise CollectionError("Statistics returned an unexpected response structure.")
    return message


def _collection_statistics_request(base, token, parameters, attempts=3, request_type="recorder/statistics_during_period"):
    if request_type not in {"recorder/statistics_during_period", "recorder/get_statistics_metadata"} or type(attempts) is not int or not 1 <= attempts <= 3:
        raise CollectionError("Choose an allowed read-only statistics request and at most three attempts.")
    address = ("wss" if base.startswith("https:") else "ws") + base[base.index(":"):] + "/api/websocket"
    for attempt in range(attempts):
        connection = None
        try:
            websocket.enableTrace(False)
            connection = websocket.create_connection(address, timeout=30, http_no_proxy=["*"], redirect_limit=0)
            if _collection_ws_message(connection).get("type") != "auth_required":
                raise CollectionError("The statistics endpoint did not request authentication.")
            connection.send(json.dumps({"type": "auth", "access_token": token}))
            if _collection_ws_message(connection).get("type") != "auth_ok":
                raise CollectionError("Statistics access was refused; check the runtime token and read permissions.")
            connection.send(json.dumps({**parameters, "id": 1, "type": request_type}))
            message = _collection_ws_message(connection)
            if message.get("id") != 1 or message.get("type") != "result" or message.get("success") is not True:
                raise CollectionError("The read-only statistics request failed or returned an unexpected reply.")
            expected_type = list if request_type == "recorder/get_statistics_metadata" else dict
            if not isinstance(message.get("result"), expected_type):
                raise CollectionError("Statistics returned invalid record data.")
            return message["result"]
        except CollectionError:
            raise
        except (OSError, TimeoutError, websocket.WebSocketConnectionClosedException, websocket.WebSocketTimeoutException):
            if attempt == attempts - 1:
                raise CollectionError("The statistics endpoint could not be reached within the retry limit.") from None
        except Exception:
            raise CollectionError("The statistics endpoint returned an unreadable or unsuccessful response.") from None
        finally:
            if connection is not None:
                try:
                    connection.close()
                except Exception:
                    pass
        time.sleep(min(2 ** attempt, 4))


def collect_home_assistant_statistics_metadata(endpoint, token, channels):
    base = _collection_endpoint(endpoint, token)
    _collection_channels(channels, "statistic_id")
    identifiers = [details["statistic_id"] for details in channels.values()]
    payload = _collection_statistics_request(base, token, {"statistic_ids": identifiers}, request_type="recorder/get_statistics_metadata")
    metadata = {}
    units = {None, "", "kWh", "Wh", "MWh", "J", "kJ", "MJ", "GJ", "m³", "m3", "ft³", "L", "l", "°C", "°F", "K", "%", "W", "kW", "rpm"}
    for entry in payload:
        if not isinstance(entry, dict) or entry.get("statistic_id") not in identifiers or entry["statistic_id"] in metadata:
            raise CollectionError("Statistics metadata contains an unexpected or duplicate channel.")
        unit = entry.get("statistics_unit_of_measurement", entry.get("unit_of_measurement"))
        if unit is not None and not isinstance(unit, str) or unit not in units or type(entry.get("has_sum")) is not bool or entry.get("has_mean") is not None and type(entry["has_mean"]) is not bool:
            raise CollectionError("Statistics metadata has unsupported units or measurement flags.")
        has_mean = entry.get("has_mean")
        if "mean_type" in entry:
            if type(entry["mean_type"]) is not int or entry["mean_type"] not in {0, 1, 2}:
                raise CollectionError("Statistics metadata has an unsupported mean type.")
            has_mean = entry["mean_type"] == 1
        metadata[entry["statistic_id"]] = {"unit_of_measurement": unit, "has_sum": entry["has_sum"], "has_mean": has_mean}
    if set(metadata) != set(identifiers):
        raise CollectionError("One or more requested statistics channels have no available metadata.")
    return pd.DataFrame([{"channel": alias, **metadata[details["statistic_id"]]} for alias, details in channels.items()])


def collect_home_assistant_statistics(endpoint, token, channels, start, end, chunk_days=7, period="hour"):
    base = _collection_endpoint(endpoint, token)
    _collection_channels(channels, "statistic_id")
    start, end = _collection_window(start, end, chunk_days)
    if period not in {"5minute", "hour"}:
        raise CollectionError("Collect five-minute or hourly statistics before forming daily totals.")
    interval = pd.Timedelta(minutes=5 if period == "5minute" else 60)
    if start.value % interval.value or end.value % interval.value:
        raise CollectionError("Align both collection boundaries to the chosen statistics interval in UTC.")
    metadata = collect_home_assistant_statistics_metadata(endpoint, token, channels).set_index("channel")
    unit_classes = {"kWh": "energy", "Wh": "energy", "MWh": "energy", "J": "energy", "kJ": "energy", "MJ": "energy", "GJ": "energy", "m³": "volume", "m3": "volume", "ft³": "volume", "L": "volume", "l": "volume", "°C": "temperature", "°F": "temperature", "K": "temperature", "W": "power", "kW": "power"}
    records = []
    for alias, details in channels.items():
        cursor = start
        while cursor < end:
            stop = min(cursor + pd.Timedelta(days=chunk_days), end)
            parameters = {"start_time": cursor.isoformat(), "end_time": stop.isoformat(), "statistic_ids": [details["statistic_id"]], "period": period, "types": ["mean", "min", "max", "state", "sum", "change"]}
            unit = metadata.loc[alias, "unit_of_measurement"]
            if unit in unit_classes:
                parameters["units"] = {unit_classes[unit]: unit}
            payload = _collection_statistics_request(base, token, parameters)
            if set(payload) - {details["statistic_id"]}:
                raise CollectionError("Statistics returned an unrequested channel.")
            points = payload.get(details["statistic_id"], [])
            if not isinstance(points, list):
                raise CollectionError("Statistics returned an invalid observation list.")
            for point in points:
                if not isinstance(point, dict):
                    raise CollectionError("Statistics returned an invalid observation.")
                try:
                    if any(type(point.get(name)) not in {int, float} for name in ("start", "end")):
                        raise ValueError
                    stamp = pd.to_datetime(point["start"], unit="ms", utc=True)
                    finish = pd.to_datetime(point["end"], unit="ms", utc=True)
                    if pd.isna(stamp) or pd.isna(finish) or finish - stamp != interval:
                        raise ValueError
                except Exception:
                    raise CollectionError("Statistics returned invalid UTC interval boundaries.") from None
                if not start <= stamp < end or finish > end:
                    continue
                row = {"channel": alias, "timestamp": stamp, "interval_end": finish, **metadata.loc[alias].to_dict()}
                for name in ("mean", "min", "max", "state", "sum", "change"):
                    value = point.get(name)
                    if value is not None and (type(value) not in {int, float} or not math.isfinite(value)):
                        raise CollectionError("Statistics returned a non-numeric measurement.")
                    row[name] = float("nan") if value is None else float(value)
                records.append(row)
            cursor = stop
    result = pd.DataFrame(records, columns=["channel", "timestamp", "interval_end", "mean", "min", "max", "state", "sum", "change", "unit_of_measurement", "has_sum", "has_mean"])
    for name in ("timestamp", "interval_end"):
        result[name] = pd.to_datetime(result[name], utc=True)
    result = result.drop_duplicates().sort_values(["channel", "timestamp"], kind="stable").reset_index(drop=True)
    if result.duplicated(["channel", "timestamp"]).any():
        raise CollectionError("Conflicting statistics share an interval; no conflicting value was selected.")
    return result


In [ ]:
def collect_era5_weather(latitude, longitude, home_alias, start, end, chunk_days=31):
    start, end = _collection_window(start, end, chunk_days)
    if not isinstance(home_alias, str) or not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,63}", home_alias):
        raise CollectionError("Use an anonymous home label for weather observations.")
    try:
        if isinstance(latitude, bool) or isinstance(longitude, bool):
            raise ValueError
        latitude, longitude = float(latitude), float(longitude)
        if not math.isfinite(latitude) or not math.isfinite(longitude) or not -90 <= latitude <= 90 or not -180 <= longitude <= 180:
            raise ValueError
    except Exception:
        raise CollectionError("Enter valid latitude and longitude values at runtime.") from None
    if start != start.floor("h") or end != end.floor("h"):
        raise CollectionError("Align weather collection boundaries to whole UTC hours.")
    records = []
    cursor = start
    while cursor < end:
        stop = min(cursor + pd.Timedelta(days=chunk_days), end)
        parameters = {"latitude": latitude, "longitude": longitude, "start_date": cursor.date().isoformat(), "end_date": (stop - pd.Timedelta(nanoseconds=1)).date().isoformat(), "hourly": "temperature_2m", "models": "era5", "timezone": "UTC", "temperature_unit": "celsius", "timeformat": "unixtime"}
        payload = _collection_json("https://archive-api.open-meteo.com/v1/archive?" + urllib.parse.urlencode(parameters))
        if not isinstance(payload, dict) or payload.get("error") or payload.get("utc_offset_seconds") != 0:
            raise CollectionError("ERA5 weather returned an unsuccessful response or an unexpected timezone.")
        hourly = payload.get("hourly")
        units = payload.get("hourly_units")
        if not isinstance(hourly, dict) or not isinstance(units, dict) or units.get("temperature_2m") != "°C" or units.get("time") != "unixtime":
            raise CollectionError("ERA5 weather returned unexpected columns or units.")
        stamps, readings = hourly.get("time"), hourly.get("temperature_2m")
        if not isinstance(stamps, list) or not isinstance(readings, list) or len(stamps) != len(readings):
            raise CollectionError("ERA5 weather timestamps and observations are not aligned.")
        for raw_stamp, value in zip(stamps, readings):
            if type(raw_stamp) not in {int, float} or not math.isfinite(raw_stamp):
                raise CollectionError("ERA5 weather returned an invalid timestamp.")
            try:
                stamp = pd.to_datetime(raw_stamp, unit="s", utc=True)
            except Exception:
                raise CollectionError("ERA5 weather returned an invalid timestamp.") from None
            if stamp != stamp.floor("h"):
                raise CollectionError("ERA5 weather returned observations outside the hourly grid.")
            if not start <= stamp < end:
                continue
            if value is not None and (type(value) not in {int, float} or not math.isfinite(value)):
                raise CollectionError("ERA5 weather returned a non-numeric temperature.")
            records.append({"home": home_alias, "timestamp": stamp, "temperature_2m_c": float("nan") if value is None else float(value), "weather_source": "Open-Meteo ERA5", "retrospective_weather": True})
        cursor = stop
    result = pd.DataFrame(records, columns=["home", "timestamp", "temperature_2m_c", "weather_source", "retrospective_weather"])
    result["timestamp"] = pd.to_datetime(result["timestamp"], utc=True)
    result = result.drop_duplicates().sort_values("timestamp").reset_index(drop=True)
    if result.duplicated("timestamp").any():
        raise CollectionError("Conflicting ERA5 temperatures share an hour; no conflicting value was selected.")
    expected = pd.date_range(start, end, freq="h", inclusive="left")
    if not pd.DatetimeIndex(result["timestamp"]).equals(expected):
        raise CollectionError("ERA5 did not return every requested hour; use a supported historical period.")
    return result


In [ ]:
import hashlib


import json


import re


from pathlib import Path


import numpy as np


import pandas as pd


def read_utc_timestamps(values):
    for value in values.dropna():
        if pd.Timestamp(value).tzinfo is None:
            raise ValueError("Every timestamp must include UTC or its recorded timezone offset.")
    return pd.to_datetime(values, utc=True, errors="raise")


def check_collection_label(label):
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_-]{0,39}", str(label)):
        raise ValueError("Use a short anonymous label containing letters, numbers, underscores or hyphens.")
    return str(label)


def normalise_observed_series(observations, value_column="value", unit="C", exclusions=()):
    if unit not in {"C", "kWh", "setting", "state"}:
        raise ValueError("Choose an explicit supported unit; no unit conversion is inferred.")
    required = {"timestamp_utc", value_column}
    if not required.issubset(observations.columns):
        raise ValueError("The observation table is missing its timestamp or selected value column.")
    result = observations[["timestamp_utc", value_column]].copy()
    result["timestamp_utc"] = read_utc_timestamps(result["timestamp_utc"])
    if result.timestamp_utc.isna().any():
        raise ValueError("Observation timestamps cannot be missing.")
    result = result.rename(columns={value_column: "value"}).sort_values("timestamp_utc")
    result = result.drop_duplicates().reset_index(drop=True)
    if result.timestamp_utc.duplicated().any():
        raise ValueError("Different values share an observation timestamp; resolve the source conflict first.")
    result["quality"] = "retained"
    if unit != "state":
        result["value"] = pd.to_numeric(result["value"], errors="coerce")
        result.loc[~np.isfinite(result.value), "quality"] = "missing_or_nonnumeric"
    for exclusion in exclusions:
        bounds = read_utc_timestamps(pd.Series([exclusion["start_utc"], exclusion["end_utc"]]))
        if bounds.iloc[1] <= bounds.iloc[0] or not str(exclusion.get("reason", "")).strip():
            raise ValueError("Each exclusion needs an ordered time window and a reason.")
        selected = result.timestamp_utc.ge(bounds.iloc[0]) & result.timestamp_utc.lt(bounds.iloc[1])
        result.loc[selected, "quality"] = "explicit_exclusion"
    result["unit"] = unit
    return result



In [ ]:
def prepare_gas_observations(observations, source_type, unit, interval_minutes, exclusions=(), maximum_interval_kwh=None):
    if unit != "kWh":
        raise ValueError("Gas must already be supplied in kWh; meter-volume conversion needs separately documented inputs.")
    if source_type not in {"interval_totals", "cumulative_meter"}:
        raise ValueError("Choose interval_totals or cumulative_meter explicitly.")
    if not isinstance(interval_minutes, int) or interval_minutes not in {30, 60}:
        raise ValueError("Choose the supported recorded interval: 30 or 60 minutes.")
    if maximum_interval_kwh is not None and (not np.isfinite(maximum_interval_kwh) or maximum_interval_kwh <= 0):
        raise ValueError("An optional upper energy limit must be positive.")
    if source_type == "interval_totals":
        if not {"interval_start", "interval_end", "kwh"}.issubset(observations.columns):
            raise ValueError("Interval gas needs interval_start, interval_end and kwh columns.")
        result = observations[["interval_start", "interval_end", "kwh"]].copy()
        for column in ("interval_start", "interval_end"):
            result[column] = read_utc_timestamps(result[column])
        result["kwh"] = pd.to_numeric(result.kwh, errors="coerce")
        result = result.drop_duplicates().sort_values(["interval_start", "interval_end"]).reset_index(drop=True)
        if result.interval_start.isna().any() or result.interval_end.isna().any():
            raise ValueError("Gas interval boundaries cannot be missing.")
        if result.interval_start.duplicated().any():
            raise ValueError("Different gas observations share an interval start; resolve the source conflict first.")
        result["quality"] = "retained"
    else:
        readings = normalise_observed_series(observations, unit="kWh")
        if len(readings) < 2:
            raise ValueError("Cumulative gas needs at least two timestamped readings.")
        result = pd.DataFrame({
            "interval_start": readings.timestamp_utc.shift(),
            "interval_end": readings.timestamp_utc,
            "kwh": readings.value.diff(),
            "quality": "retained",
        })
        result.loc[result.interval_start.isna(), "quality"] = "no_previous_reading"
    valid_energy = np.isfinite(result.kwh)
    result.loc[result.quality.eq("retained") & ~valid_energy, "quality"] = "missing_energy"
    result.loc[result.quality.eq("retained") & result.kwh.lt(0), "quality"] = "negative_increment_or_reset"
    duration = result.interval_end - result.interval_start
    result.loc[result.quality.eq("retained") & duration.ne(pd.Timedelta(minutes=interval_minutes)), "quality"] = "unexpected_interval_or_gap"
    if maximum_interval_kwh is not None:
        result.loc[result.quality.eq("retained") & result.kwh.ge(maximum_interval_kwh), "quality"] = "explicit_upper_limit"
    ordered = result.loc[result.interval_start.notna() & result.interval_end.gt(result.interval_start)].sort_values("interval_start")
    previous_latest_end = ordered.interval_end.cummax().shift()
    if ordered.interval_start.lt(previous_latest_end).any():
        raise ValueError("Gas intervals overlap; resolve the overlapping source rows before aggregation.")
    for exclusion in exclusions:
        bounds = read_utc_timestamps(pd.Series([exclusion["start_utc"], exclusion["end_utc"]]))
        if bounds.iloc[1] <= bounds.iloc[0] or not str(exclusion.get("reason", "")).strip():
            raise ValueError("Each exclusion needs an ordered time window and a reason.")
        overlap = result.interval_start.lt(bounds.iloc[1]) & result.interval_end.gt(bounds.iloc[0])
        result.loc[overlap, "quality"] = "explicit_exclusion"
    result["source_type"] = source_type
    result["unit"] = "kWh"
    return result.reset_index(drop=True)



In [ ]:
def local_day_expected_hours(dates, timezone="Europe/London"):
    dates = pd.DatetimeIndex(pd.to_datetime(dates))
    starts = dates.tz_localize(timezone)
    ends = (dates + pd.Timedelta(days=1)).tz_localize(timezone)
    return pd.Series((ends - starts).total_seconds() / 3600, index=dates, name="expected_hours")


def summarise_daily_gas(gas_intervals, interval_minutes, minimum_observations, excluded_dates=(), timezone="Europe/London"):
    if not isinstance(minimum_observations, int) or minimum_observations < 1:
        raise ValueError("The minimum contributing-observation count must be a positive integer.")
    result = gas_intervals.copy()
    first_dates = result.interval_start.dt.tz_convert(timezone).dt.tz_localize(None).dt.normalize()
    final_dates = (result.interval_end - pd.Timedelta(nanoseconds=1)).dt.tz_convert(timezone).dt.tz_localize(None).dt.normalize()
    crosses_day = first_dates.notna() & first_dates.ne(final_dates)
    result.loc[result.quality.eq("retained") & crosses_day, "quality"] = "crosses_local_midnight"
    result["date"] = first_dates
    dated = result.loc[result.date.notna()]
    if dated.empty:
        raise ValueError("No dated gas intervals are available.")
    dates = pd.date_range(dated.date.min(), dated.date.max(), freq="D")
    retained = dated.loc[dated.quality.eq("retained")]
    daily = retained.groupby("date").kwh.agg(gas_kwh=lambda values: values.sum(min_count=1), observations="count").reindex(dates)
    daily["observations"] = daily.observations.fillna(0).astype(int)
    daily["expected_observations"] = (local_day_expected_hours(dates, timezone) * 60 / interval_minutes).astype(int)
    daily["available_fraction"] = daily.observations / daily.expected_observations
    daily["excluded_date"] = dates.strftime("%Y-%m-%d").isin([str(pd.Timestamp(date).date()) for date in excluded_dates])
    daily["eligible"] = daily.observations.ge(minimum_observations) & daily.gas_kwh.notna() & ~daily.excluded_date
    daily.index.name = "date"
    return daily.reset_index(), result



In [ ]:
def prepare_hourly_weather(observations, exclusions=()):
    observations = normalise_observed_series(observations, value_column="temperature_c", unit="C", exclusions=exclusions)
    if observations.empty:
        raise ValueError("At least one timestamped weather observation is required.")
    values = observations.set_index("timestamp_utc").value.where(observations.set_index("timestamp_utc").quality.eq("retained"))
    hourly = values.resample("h").agg(temperature_c="mean", observations="count")
    hourly.index.name = "timestamp_utc"
    return hourly.reset_index(), observations


def summarise_daily_weather(hourly_weather, reference_temperatures, minimum_weather_hours=20, timezone="Europe/London"):
    references = np.asarray(reference_temperatures, dtype=float)
    if references.ndim != 1 or not len(references) or not np.isfinite(references).all() or len(np.unique(references)) != len(references):
        raise ValueError("Reference temperatures must be distinct finite numbers.")
    if not isinstance(minimum_weather_hours, int) or minimum_weather_hours < 1:
        raise ValueError("The weather-hour threshold must be a positive integer.")
    values = hourly_weather.copy()
    values["timestamp_utc"] = read_utc_timestamps(values.timestamp_utc)
    if values.timestamp_utc.duplicated().any() or values.timestamp_utc.ne(values.timestamp_utc.dt.floor("h")).any():
        raise ValueError("Weather needs one row per UTC hour before daily aggregation.")
    values["temperature_c"] = pd.to_numeric(values.temperature_c, errors="coerce")
    values.loc[~np.isfinite(values.temperature_c), "temperature_c"] = np.nan
    dates = values.timestamp_utc.dt.tz_convert(timezone).dt.tz_localize(None).dt.normalize()
    observed = values.temperature_c.notna()
    shortfalls = np.maximum(0.0, references[None, :] - values.loc[observed, "temperature_c"].to_numpy()[:, None])
    degree_days = pd.DataFrame(shortfalls, columns=references, index=pd.DatetimeIndex(dates.loc[observed], name="date")).groupby(level=0).mean()
    counts = values.assign(date=dates).groupby("date").temperature_c.agg(weather_hours="count", mean_temperature_c="mean")
    counts["expected_hours"] = local_day_expected_hours(counts.index, timezone).astype(int)
    counts["available_fraction"] = counts.weather_hours / counts.expected_hours
    counts["eligible"] = counts.weather_hours.ge(minimum_weather_hours)
    degree_days = degree_days.reindex(counts.index)
    degree_days.columns = [f"hdd_{reference:g}" for reference in references]
    return degree_days.reset_index(), counts.reset_index()



In [ ]:
def prepare_household_collection(home, gas_observations, weather_observations, source_type, gas_unit, interval_minutes, minimum_gas_observations, reference_temperatures, minimum_weather_hours=20, excluded_dates=(), gas_exclusions=(), weather_exclusions=(), maximum_interval_kwh=None):
    home = check_collection_label(home)
    gas = prepare_gas_observations(gas_observations, source_type, gas_unit, interval_minutes, gas_exclusions, maximum_interval_kwh)
    daily_gas, gas = summarise_daily_gas(gas, interval_minutes, minimum_gas_observations, excluded_dates)
    hourly_weather, weather = prepare_hourly_weather(weather_observations, weather_exclusions)
    degree_days, daily_weather = summarise_daily_weather(hourly_weather, reference_temperatures, minimum_weather_hours)
    matched = daily_gas.rename(columns={"eligible": "gas_eligible"}).merge(daily_weather.rename(columns={"eligible": "weather_eligible", "available_fraction": "weather_available_fraction"}), on="date", how="outer")
    matched = matched.merge(degree_days, on="date", how="left").sort_values("date")
    matched["eligible"] = matched.gas_eligible.fillna(False).astype(bool) & matched.weather_eligible.fillna(False).astype(bool)
    summary = {
        "home": home,
        "gas_source_type": source_type,
        "gas_unit": gas_unit,
        "gas_interval_minutes": interval_minutes,
        "minimum_gas_observations": minimum_gas_observations,
        "minimum_weather_hours": minimum_weather_hours,
        "timezone": "Europe/London",
        "negative_gas_changes": "excluded; never replaced by zero",
        "irregular_gas_intervals": "excluded; no energy apportioned across gaps or local midnight",
        "hourly_weather": "mean of recorded observations within each UTC hour; no interpolation",
        "degree_days": "mean positive hourly temperature shortfall within each local date",
        "excluded_dates": [str(pd.Timestamp(date).date()) for date in excluded_dates],
        "gas_exclusions": list(gas_exclusions),
        "weather_exclusions": list(weather_exclusions),
        "maximum_interval_kwh": maximum_interval_kwh,
        "matched_eligible_dates": int(matched.eligible.sum()),
        "gas_quality_counts": {str(key): int(value) for key, value in gas.quality.value_counts().items()},
    }
    return {"home": home, "gas_intervals": gas, "weather_observations": weather, "hourly_weather": hourly_weather, "daily_gas": daily_gas, "daily_weather": daily_weather, "degree_days": degree_days, "daily_comparison": matched, "summary": summary}



In [ ]:
def write_household_collection(prepared, collection_folder, additional_observations=None):
    home = check_collection_label(prepared["home"])
    folder = Path(collection_folder) / home
    folder.mkdir(parents=True, exist_ok=False)
    saved_files = []
    for name in ("gas_intervals", "weather_observations", "hourly_weather", "daily_gas", "daily_weather", "degree_days", "daily_comparison"):
        target = folder / f"{name}.csv"
        prepared[name].to_csv(target, index=False)
        saved_files.append(target)
    (folder / "preparation.json").write_text(json.dumps(prepared["summary"], indent=2), encoding="utf-8")
    saved_files.append(folder / "preparation.json")
    if additional_observations:
        archive = folder / "observations"
        archive.mkdir()
        for label, observations in additional_observations.items():
            label = check_collection_label(label)
            if list(observations.columns) != ["timestamp_utc", "value", "quality", "unit"]:
                raise ValueError("Additional series must use the normalised anonymous observation structure.")
            target = archive / f"{label}.csv"
            observations.to_csv(target, index=False)
            saved_files.append(target)
    manifest = [{"file": str(path.relative_to(folder)), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in saved_files]
    (folder / "input_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return folder



In [ ]:
def load_new_collection_model_inputs(collection_folder, home, reference_temperatures):
    folder = Path(collection_folder) / check_collection_label(home)
    manifest = json.loads((folder / "input_manifest.json").read_text(encoding="utf-8"))
    listed_files = [item["file"] for item in manifest]
    if len(set(listed_files)) != len(listed_files) or not {"daily_comparison.csv", "hourly_weather.csv", "preparation.json"}.issubset(listed_files):
        raise ValueError("The dataset manifest is incomplete or repeats file entries.")
    for item in manifest:
        relative = Path(item["file"])
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("The dataset manifest contains an unsafe file path.")
        path = folder / relative
        if path.is_symlink() or folder.resolve() not in path.resolve().parents:
            raise ValueError("The dataset manifest points outside its dataset folder.")
        if hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("A collected input has changed since preparation.")
    combined = pd.read_csv(folder / "daily_comparison.csv", parse_dates=["date"], float_precision="round_trip")
    combined = combined.loc[combined.eligible.eq(True)].set_index("date").sort_index()
    references = np.asarray(reference_temperatures, dtype=float)
    names = [f"hdd_{reference:g}" for reference in references]
    if combined.empty or not set(names).issubset(combined.columns):
        raise ValueError("There are no eligible paired days or the requested reference temperatures were not prepared.")
    demand = combined[names].copy()
    demand.columns = references
    if not combined.index.is_unique or not np.isfinite(demand.to_numpy()).all() or not np.isfinite(combined.gas_kwh).all() or combined.gas_kwh.lt(0).any():
        raise ValueError("The paired days contain duplicate dates or invalid modelling values.")
    weather = pd.read_csv(folder / "hourly_weather.csv", float_precision="round_trip")
    weather.index = pd.to_datetime(weather.timestamp_utc, utc=True).dt.tz_convert("Europe/London")
    return {"gas": combined.gas_kwh, "degree_days": demand, "temperature": weather.temperature_c, "weather": "Collected outdoor observations"}



In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import shutil
import uuid

ARCHIVE_INPUT_GROUPS = {
    "inputs/observations": ["monthly_availability.csv", "h2_flow_selected.csv", "h2_return_selected.csv", "h2_gas_selected.csv"],
    "inputs/daily": ["h1_daily_gas.csv", "h2_daily_gas.csv", "h1_hourly_weather.csv", "h2_hourly_weather.csv", "legacy_score_summary.csv"],
    "inputs/comparison": ["h1_hourly_inputs.parquet"],
    "inputs/sensor_value": ["h2_flow.parquet", "h2_return.parquet", "h2_gas_intervals.parquet"],
    "inputs/earlier_experiments/events": ["h1_flow.parquet", "h1_reference.parquet", "h2_flow.parquet", "h2_gas_intervals.parquet"],
    "inputs/earlier_experiments/signatures": ["h1_daily_gas.csv", "h2_daily_gas.csv", "h1_hourly_weather.csv", "h2_hourly_weather.csv"],
    "inputs/earlier_experiments/scenarios": ["h1_signature_inputs.parquet", "h2_signature_inputs.parquet", "model_parameters.json"],
    "supplementary/h1_settings/inputs": ["daily_settings.csv", "hourly_weather.csv", "pump_helper_regimes.csv", "pipe_coverage_by_pump_regime.csv", "expected_pump_summary.csv", "expected_primary_model_scores.csv", "thermostat_target_changes.csv", "observation_metadata.json"],
}
ARCHIVE_ROOM_SENSORS = [
    "H1_living_room", "H1_dining_room", "H1_kitchen", "H1_office", "H1_master_bedroom", "H1_bedroom_1",
    "H2_living_room", "H2_playroom", "H2_kitchen", "H2_bathroom", "H2_bedroom_1", "H2_master_bedroom", "H1_outdoor", "H2_outdoor",
]




In [ ]:
def archive_file(root, relative_name):
    relative = PurePosixPath(relative_name)
    if relative.is_absolute() or ".." in relative.parts or "\\" in relative_name:
        raise ValueError("The archive contains an unsafe relative file name.")
    candidate = root.joinpath(*relative.parts)
    if not candidate.resolve().is_relative_to(root) or not candidate.is_file():
        raise ValueError("The archive is missing a required input or points outside its folder.")
    return candidate


In [ ]:
def archive_digest(contents):
    return hashlib.sha256(contents).hexdigest()


In [ ]:
def archive_json(contents):
    return (json.dumps(contents, indent=2, ensure_ascii=False) + "\n").encode("utf-8")


In [ ]:
def inspect_prepared_archive(source_folder):
    root = Path(source_folder).expanduser().resolve()
    if not root.is_dir():
        raise ValueError("Choose an existing analysis or prepared-input bundle folder.")
    contents, origins, coverage = {}, {}, []
    for group, names in ARCHIVE_INPUT_GROUPS.items():
        manifest_name = "manifest.json" if group.endswith("h1_settings/inputs") else "input_manifest.json"
        relative_manifest = f"{group}/{manifest_name}"
        raw_manifest = archive_file(root, relative_manifest).read_bytes()
        manifest = json.loads(raw_manifest)
        entries = manifest if isinstance(manifest, list) else manifest.get("inputs", manifest.get("files"))
        if not isinstance(entries, list):
            raise ValueError("A prepared-input manifest has an unsupported structure.")
        selected = []
        for name in names:
            matches = [entry for entry in entries if entry.get("file", entry.get("input_file")) == name]
            if len(matches) != 1:
                raise ValueError("A required input must have exactly one manifest entry.")
            item = matches[0]
            relative_name = f"{group}/{name}"
            original = archive_file(root, relative_name).read_bytes()
            original_digest = archive_digest(original)
            if original_digest != item.get("sha256"):
                raise ValueError("A prepared input no longer matches its recorded SHA-256 hash.")
            copied = original
            if name == "observation_metadata.json":
                metadata = json.loads(original)
                copied = archive_json({"observed_target_change_count": metadata["observed_target_change_count"]})
            if name == "model_parameters.json":
                parameters = json.loads(original)
                copied = archive_json({key: parameters[key] for key in ("constants", "pump_dt_K", "room_reference_C", "emitter", "tariff_p_per_kwh")})
            contents[relative_name] = copied
            origins[relative_name] = original_digest
            file_key = "input_file" if "input_file" in item else "file"
            prepared = {file_key: name, "sha256": archive_digest(copied)}
            if "rows" in item:
                prepared["rows"] = item["rows"]
            if group.endswith("/events"):
                prepared["role"] = {
                    "h1_flow.parquet": "H1 flow temperature",
                    "h1_reference.parquet": "H1 request for heat",
                    "h2_flow.parquet": "H2 flow temperature",
                    "h2_gas_intervals.parquet": "H2 gas consumption",
                }[name]
            selected.append(prepared)
        if isinstance(manifest, list):
            cleaned_manifest = selected
        elif "inputs" in manifest:
            cleaned_manifest = {"inputs": selected}
        else:
            cleaned_manifest = {"files": selected}
        if group == "inputs/observations":
            cleaned_manifest["trace_window_utc"] = manifest["trace_window_utc"]
        contents[relative_manifest] = archive_json(cleaned_manifest)
        origins[relative_manifest] = archive_digest(raw_manifest)
        coverage.append({"group": group, "input_files": len(names), "hashes_verified": len(names)})
    room_group = "supplementary/sensor_relationships"
    relative_manifest = f"{room_group}/inputs/input_manifest.json"
    raw_manifest = archive_file(root, relative_manifest).read_bytes()
    room_manifest = json.loads(raw_manifest)
    selected = []
    for alias in ARCHIVE_ROOM_SENSORS:
        matches = [item for item in room_manifest["files"] if item.get("alias") == alias]
        if len(matches) != 1:
            raise ValueError("A required room sensor must have exactly one manifest entry.")
        item = matches[0]
        expected_path = f"inputs/{alias}.parquet"
        if item["path"] != expected_path:
            raise ValueError("A room-sensor input uses an unexpected file name.")
        relative_name = f"{room_group}/{expected_path}"
        original = archive_file(root, relative_name).read_bytes()
        digest = archive_digest(original)
        if digest != item.get("sha256"):
            raise ValueError("A room-sensor input no longer matches its recorded SHA-256 hash.")
        contents[relative_name] = original
        origins[relative_name] = digest
        selected.append({"alias": alias, "path": expected_path, "sha256": digest})
    contents[relative_manifest] = archive_json({"files": selected})
    origins[relative_manifest] = archive_digest(raw_manifest)
    coverage.append({"group": f"{room_group}/inputs", "input_files": len(selected), "hashes_verified": len(selected)})
    if "bundle_manifest.json" in {item.name for item in root.iterdir()}:
        bundle_manifest = json.loads(archive_file(root, "bundle_manifest.json").read_bytes())
        bundle_files = {item["path"]: item["sha256"] for item in bundle_manifest["files"]}
        for relative_name in contents:
            actual = archive_digest(archive_file(root, relative_name).read_bytes())
            if bundle_files.get(relative_name) != actual:
                raise ValueError("The prepared archive does not match its bundle manifest.")
    return {"contents": contents, "original_hashes": origins, "coverage": coverage}


In [ ]:
def import_prepared_archive(source_folder, destination_parent):
    prepared = inspect_prepared_archive(source_folder)
    parent = Path(destination_parent).expanduser().resolve()
    if not parent.is_dir():
        raise ValueError("Choose an existing destination folder.")
    name = datetime.now(timezone.utc).strftime("prepared-observations-%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:8]
    destination = parent / name
    destination.mkdir(mode=0o700, exist_ok=False)
    try:
        for relative_name, contents in prepared["contents"].items():
            target = destination / relative_name
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open("xb") as handle:
                handle.write(contents)
            target.chmod(0o600)
        provenance = {
            "format": "from_sensor_to_insight_prepared_inputs_v1",
            "mode": "archived_prepared_input_replay",
            "scope": "Prepared observations and historical scenario parameters for the study analysis; not raw collection or proof of sensor accuracy.",
            "connection_details_saved": False,
            "input_files_verified": sum(group["hashes_verified"] for group in prepared["coverage"]),
            "groups": prepared["coverage"],
            "files": [{"path": name, "sha256": archive_digest(contents), "source_sha256": prepared["original_hashes"][name], "bytes": len(contents), "metadata_reduced": archive_digest(contents) != prepared["original_hashes"][name]} for name, contents in sorted(prepared["contents"].items())],
        }
        manifest_path = destination / "bundle_manifest.json"
        with manifest_path.open("xb") as handle:
            handle.write(archive_json(provenance))
        manifest_path.chmod(0o600)
        run_record = {
            "format_version": 1,
            "profile": "study_replay",
            "source_mode": "prepared_archive",
            "prepared_input_format": "from_sensor_to_insight_prepared_inputs_v1",
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "raw_collection_performed": False,
            "connection_details_saved": False,
            "input_files_verified": provenance["input_files_verified"],
            "provenance_manifest": "bundle_manifest.json",
            "scope": "Historical study replay from already-prepared inputs, including archived scenario parameters.",
        }
        run_path = destination / "collection_run.json"
        with run_path.open("xb") as handle:
            handle.write(archive_json(run_record))
        run_path.chmod(0o600)
        inspect_prepared_archive(destination)
    except Exception:
        shutil.rmtree(destination)
        raise RuntimeError("Archive import failed; the incomplete new folder was removed.") from None
    return destination


In [ ]:
import getpass

In [ ]:
import warnings

In [ ]:
from datetime import datetime, timezone

In [ ]:
import uuid

In [ ]:
import nbformat

In [ ]:
def private_response(prompt, default=None):
    with warnings.catch_warnings():
        warnings.simplefilter("error", getpass.GetPassWarning)
        response = getpass.getpass(prompt).strip()
    return str(default) if not response and default is not None else response

In [ ]:
def choose_response(prompt, choices, default):
    value = input(prompt).strip() or default
    if value not in choices:
        raise ValueError("Choose one of the displayed options.")
    return value

In [ ]:
def choose_number(prompt, default, minimum, maximum):
    value = int(input(prompt).strip() or default)
    if not minimum <= value <= maximum:
        raise ValueError("The number is outside the displayed range.")
    return value

In [ ]:
def collection_date(prompt):
    value = pd.Timestamp(input(prompt).strip())
    if value.tzinfo is None:
        value = value.tz_localize("UTC")
    return _collection_time(value)

In [ ]:
def read_observation_export(prompt, columns):
    path = Path(private_response(prompt)).expanduser()
    if path.suffix.lower() == ".csv":
        table = pd.read_csv(path, float_precision="round_trip")
    elif path.suffix.lower() == ".parquet":
        table = pd.read_parquet(path)
    else:
        raise ValueError("Use a CSV or Parquet observation export.")
    if not set(columns).issubset(table.columns):
        raise ValueError("The observation export does not contain the required columns.")
    return table[list(columns)].copy()

In [ ]:
def trim_observation_period(table, column, start, end, include_end=False):
    result = table.copy()
    result[column] = read_utc_timestamps(result[column])
    selected = result[column].ge(start) & result[column].lt(end)
    if include_end:
        selected = result[column].ge(start) & result[column].le(end)
    return result.loc[selected].sort_values(column).reset_index(drop=True)

In [ ]:
def collect_home_observations(home, mode, start, end, references):
    endpoint = token = gas_identifier = weather_identifier = latitude = longitude = identifier = attribute = channel = None
    try:
        print(f"Preparing {home}. Source identifiers and connection details are hidden and are not saved.")
        gas_choices = {"1", "2", "3"} if mode == "live" else {"2", "3"}
        gas_source = choose_response("Gas: 1 Home Assistant hourly cumulative statistics, 2 interval export, 3 cumulative export [2]: ", gas_choices, "2")
        weather_choices = {"1", "2", "3"} if mode == "live" else {"3"}
        weather_source = choose_response("Weather: 1 Home Assistant hourly temperature statistics, 2 ERA5 historical weather, 3 observation export [3]: ", weather_choices, "3")
        extra_count = choose_number("Additional pipe, room, thermostat or pump series [0; maximum 30]: ", 0, 0, 30)
        if mode == "live" and (gas_source == "1" or weather_source == "1" or extra_count):
            endpoint = private_response("Home Assistant endpoint reachable through your existing Tailscale connection (hidden): ")
            token = private_response("Home Assistant access token (hidden): ")
            _collection_endpoint(endpoint, token)
        if gas_source == "1":
            gas_identifier = private_response("Cumulative gas statistic identifier (hidden; recorded energy must be kWh): ")
            gas_field = choose_response("Cumulative statistic to use: sum or state [sum]: ", {"sum", "state"}, "sum")
            recorded_gas = collect_home_assistant_statistics(endpoint, token, {"gas": {"statistic_id": gas_identifier}}, start - pd.Timedelta(hours=1), end)
            if recorded_gas.empty or set(recorded_gas.unit_of_measurement.dropna()) != {"kWh"}:
                raise ValueError("The selected gas statistic must contain recorded energy in kWh.")
            if gas_field == "sum" and not recorded_gas.has_sum.eq(True).all():
                raise ValueError("The selected statistic does not provide cumulative energy sums.")
            gas = recorded_gas[["interval_end", gas_field]].rename(columns={"interval_end": "timestamp_utc", gas_field: "value"})
            source_type, interval_minutes = "cumulative_meter", 60
        elif gas_source == "2":
            print("Gas export columns: interval_start, interval_end, kwh; timestamps need timezone offsets.")
            gas = read_observation_export("Path to the gas interval export (hidden): ", ["interval_start", "interval_end", "kwh"])
            source_type = "interval_totals"
            interval_minutes = int(choose_response("Recorded interval in minutes: 30 or 60 [30]: ", {"30", "60"}, "30"))
        else:
            print("Cumulative export columns: timestamp_utc, value; value must be cumulative kWh at regular recorded boundaries.")
            gas = read_observation_export("Path to the cumulative gas export (hidden): ", ["timestamp_utc", "value"])
            source_type = "cumulative_meter"
            interval_minutes = int(choose_response("Recorded interval in minutes: 30 or 60 [60]: ", {"30", "60"}, "60"))
        if source_type == "interval_totals":
            gas = trim_observation_period(gas, "interval_start", start, end)
            gas["interval_end"] = read_utc_timestamps(gas.interval_end)
            gas = gas.loc[gas.interval_end.le(end)].copy()
        else:
            gas = trim_observation_period(gas, "timestamp_utc", start, end, include_end=True)
        minimum_gas = choose_number("Minimum gas intervals per local day [46 for half-hours; 20 for hours]: ", 46 if interval_minutes == 30 else 20, 1, 50 if interval_minutes == 30 else 25)
        if weather_source == "1":
            weather_identifier = private_response("Outdoor temperature statistic identifier (hidden; recorded temperature must be Celsius): ")
            recorded_weather = collect_home_assistant_statistics(endpoint, token, {"outdoor": {"statistic_id": weather_identifier}}, start, end)
            if recorded_weather.empty or not recorded_weather.unit_of_measurement.isin(["°C", "C"]).all() or not recorded_weather.has_mean.eq(True).all():
                raise ValueError("The selected weather statistic must provide hourly mean Celsius temperatures.")
            weather = recorded_weather[["timestamp", "mean"]].rename(columns={"timestamp": "timestamp_utc", "mean": "temperature_c"})
            weather_description = "Home Assistant hourly mean outdoor temperature"
        elif weather_source == "2":
            latitude = float(private_response("Weather latitude (hidden; sent only to the weather service): "))
            longitude = float(private_response("Weather longitude (hidden; sent only to the weather service): "))
            recorded_weather = collect_era5_weather(latitude, longitude, home, start, end)
            latitude = longitude = None
            weather = recorded_weather[["timestamp", "temperature_2m_c"]].rename(columns={"timestamp": "timestamp_utc", "temperature_2m_c": "temperature_c"})
            weather_description = "Open-Meteo ERA5 historical reanalysis; not an issue-time forecast"
        else:
            print("Weather export columns: timestamp_utc, temperature_c; recorded temperatures in Celsius.")
            weather = read_observation_export("Path to the outdoor weather export (hidden): ", ["timestamp_utc", "temperature_c"])
            weather_description = "Archived outdoor observations; original observation times retained"
        weather = trim_observation_period(weather, "timestamp_utc", start, end)
        minimum_weather = choose_number("Minimum observed weather hours per local day [20]: ", 20, 1, 25)
        excluded_text = input("Local dates to exclude, comma-separated YYYY-MM-DD; blank for none: ").strip()
        excluded_dates = [str(pd.Timestamp(value.strip()).date()) for value in excluded_text.split(",") if value.strip()]
        additional = {}
        roles = {"flow_pipe": "C", "return_pipe": "C", "room_temperature": "C", "thermostat_target": "C", "thermostat_air": "C", "pump_setting": "setting", "heating_request": "state", "hot_water_request": "state"}
        for number in range(1, extra_count + 1):
            role = choose_response("Series role: flow_pipe, return_pipe, room_temperature, thermostat_target, thermostat_air, pump_setting, heating_request or hot_water_request [room_temperature]: ", set(roles), "room_temperature")
            alias = f"{role}_{number:02d}"
            if mode == "live":
                identifier = private_response("Entity identifier for this series (hidden): ")
                attribute = private_response("Numeric attribute name, or blank to use the entity state (hidden): ") or None
                if roles[role] == "state" and attribute is not None:
                    raise ValueError("Operational request series use the recorded state, not a numeric attribute.")
                if roles[role] == "C":
                    choose_response("Confirm this source's recorded values are Celsius by entering C: ", {"C"}, "")
                channel = {alias: {"entity_id": identifier, "attribute": attribute}}
                history = collect_home_assistant_history(endpoint, token, channel, start, end)
                value_column = "state" if roles[role] == "state" else "value"
                observations = history[["timestamp", value_column]].rename(columns={"timestamp": "timestamp_utc", value_column: "value"})
                identifier = attribute = channel = None
            else:
                print("Additional export columns: timestamp_utc, value; temperatures must be Celsius and pump values are logged settings.")
                observations = read_observation_export("Path to this observation export (hidden): ", ["timestamp_utc", "value"])
                if roles[role] == "state":
                    observations["value"] = observations.value.map(lambda value: _collection_state(value)[0])
            observations = trim_observation_period(observations, "timestamp_utc", start, end)
            additional[alias] = normalise_observed_series(observations, unit=roles[role])
        prepared = prepare_household_collection(home, gas, weather, source_type, "kWh", interval_minutes, minimum_gas, references, minimum_weather_hours=minimum_weather, excluded_dates=excluded_dates)
        prepared["summary"]["weather_source"] = weather_description
        prepared["summary"]["requested_start_utc"] = start.isoformat()
        prepared["summary"]["requested_end_utc_exclusive"] = end.isoformat()
        return prepared, additional
    finally:
        endpoint = token = gas_identifier = weather_identifier = latitude = longitude = identifier = attribute = channel = None

In [ ]:
def copy_analysis_notebook(destination):
    candidates = [folder / "from_sensor_to_insight.ipynb" for folder in [Path.cwd(), *Path.cwd().parents]]
    source = next((path for path in candidates if path.is_file()), None)
    if source is None:
        raise ValueError("Keep from_sensor_to_insight.ipynb beside the collection notebook before running it.")
    notebook = nbformat.read(source, as_version=4)
    if any(cell.cell_type != "code" for cell in notebook.cells):
        raise ValueError("Use the code-only analysis notebook supplied with this collection notebook.")
    for cell in notebook.cells:
        cell.outputs = []
        cell.execution_count = None
        cell.metadata = {}
    notebook.metadata = {key: notebook.metadata[key] for key in ["kernelspec", "language_info", "authors", "title"] if key in notebook.metadata}
    nbformat.validate(notebook)
    target = Path(destination) / "from_sensor_to_insight.ipynb"
    if target.exists():
        raise ValueError("The analysis notebook already exists in the destination.")
    nbformat.write(notebook, target)
    target.chmod(0o600)

In [ ]:
def collect_energy_data():
    destination = None
    completed = False
    try:
        print("Choose 1 to replay the original prepared study inputs, 2 to collect new observations, or 3 to prepare existing observation exports.")
        mode = choose_response("Collection route [1]: ", {"1", "2", "3"}, "1")
        default_parent = Path.home() / "From_Sensor_to_Insight_Data"
        parent = Path(private_response("Destination parent folder (hidden; blank uses From_Sensor_to_Insight_Data in your home folder): ", default_parent)).expanduser().resolve()
        if parent.exists() and not parent.is_dir():
            raise ValueError("The destination parent must be a folder.")
        parent.mkdir(parents=True, exist_ok=True, mode=0o700)
        if mode == "1":
            source = private_response("Folder containing the original inputs and supplementary input folders (hidden): ")
            destination = import_prepared_archive(source, parent)
            source = None
        else:
            start = collection_date("Collection start in UTC, YYYY-MM-DD or timezone-aware timestamp: ")
            end = collection_date("Collection end in UTC, exclusive: ")
            _collection_window(start, end, 1)
            if start.floor("h") != start or end.floor("h") != end:
                raise ValueError("Choose whole-hour collection boundaries.")
            homes = choose_number("Number of homes [1; maximum 10]: ", 1, 1, 10)
            references = np.arange(10.0, 24.01, 0.25).tolist()
            minimum_training = choose_number("Minimum earlier eligible days before model assessment [60]: ", 60, 60, 3660)
            minimum_assessment = choose_number("Minimum eligible days in an assessment month [10]: ", 10, 1, 31)
            name = datetime.now(timezone.utc).strftime("new-observations-%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:8]
            destination = parent / name
            destination.mkdir(mode=0o700)
            home_records = []
            for number in range(1, homes + 1):
                home = f"H{number}"
                prepared, additional = collect_home_observations(home, "live" if mode == "2" else "files", start, end, references)
                folder = write_household_collection(prepared, destination / "prepared/daily", additional)
                home_records.append({"home": home, "relative_folder": str(folder.relative_to(destination)), "matched_eligible_dates": prepared["summary"]["matched_eligible_dates"], "manifest_sha256": hashlib.sha256((folder / "input_manifest.json").read_bytes()).hexdigest()})
                print(f"{home}: {prepared['summary']['matched_eligible_dates']} eligible gas/weather dates.")
            record = {"format_version": 1, "profile": "new_collection", "source_mode": "live" if mode == "2" else "observation_exports", "created_utc": datetime.now(timezone.utc).isoformat(), "start_utc": start.isoformat(), "end_utc_exclusive": end.isoformat(), "connection_details_saved": False, "homes": home_records, "reference_temperatures": references, "minimum_training_days": minimum_training, "minimum_test_days": minimum_assessment, "interpretation": "New exploratory observations; use realised weather and keep these results separate from the original paper."}
            manifest = destination / "prepared/new_collection_manifest.json"
            manifest.write_text(json.dumps(record, indent=2) + "\n")
            record["prepared_manifest_sha256"] = hashlib.sha256(manifest.read_bytes()).hexdigest()
            (destination / "collection_run.json").write_text(json.dumps(record, indent=2) + "\n")
        copy_analysis_notebook(destination)
        for path in destination.rglob("*"):
            path.chmod(0o700 if path.is_dir() else 0o600)
        completed = True
        print(f"Created {destination.name} in the destination you selected. Open its from_sensor_to_insight.ipynb and run all cells.")
        return destination
    except (KeyboardInterrupt, EOFError):
        print("Collection cancelled. No connection details were saved.")
    except CollectionError as error:
        print(f"Collection stopped: {error}")
    except Exception:
        print("Collection stopped. Check the chosen files, timestamps, units, coverage and available Python packages. No connection details were saved.")
    finally:
        if destination is not None and not completed:
            shutil.rmtree(destination, ignore_errors=True)

In [ ]:
collection_folder = collect_energy_data()